In [1]:
using CoulombIntegral

Precompiling packages...
   1470.6 ms  ✓ CoulombIntegral
  1 dependency successfully precompiled in 3 seconds. 39 already precompiled.


### Available methods:

In [2]:
using InteractiveUtils
for st in subtypes(CoulombIntegral.Method)
    println(st)
end

Expand
MonteCarlo


In [3]:
# a simple example of a wavefunction-getter function
wf = (nl)->(x->sin(nl[1]*x))

#1 (generic function with 1 method)

### Even parity integrands:

In [4]:
@time coulomb_integral(Expand(), wf,wf,(1,0,0),(1,0,0),(1,0,0),(1,0,0);
                    R=π, SH_basis=:complex) |> println
@time coulomb_integral(Expand(), wf,wf,(1,0,0),(1,0,0),(1,0,0),(1,0,0);
                    R=π, SH_basis=:real) |> println
@time coulomb_integral(MonteCarlo(100000), wf,wf,(1,0,0),(1,0,0),(1,0,0),(1,0,0);
                    R=π, SH_basis=:complex) |> println
@time coulomb_integral(MonteCarlo(100000), wf,wf,(1,0,0),(1,0,0),(1,0,0),(1,0,0);
                    R=π, SH_basis=:real) |> println

(int = 8.974001259406979, err = 1.3369370041628229e-7)
  1.444254 seconds (3.48 M allocations: 238.057 MiB, 9.98% gc time, 99.39% compilation time)
(int = 8.974001259406979, err = 1.3369370041628229e-7)
  0.093011 seconds (58.35 k allocations: 4.288 MiB, 89.91% compilation time)
(int = 8.943660672429258 + 0.0im, err = 0.05605241464360157, agg = CoulombIntegral.MonteCarloModule.Aggregator(0.0005301989443593385 + 0.0im, 0.11041584477239887, 100000))
  3.103475 seconds (14.60 M allocations: 717.423 MiB, 3.32% gc time, 56.35% compilation time)
(int = 9.012484330014924, err = 0.05807816039991343, agg = CoulombIntegral.MonteCarloModule.Aggregator(0.0005342789549875769, 0.11854096303123238, 100000))
  1.403133 seconds (11.41 M allocations: 494.749 MiB, 3.20% gc time, 10.38% compilation time)


### Odd parity integrands:

real spherical harmonics: x,y,z-reflection parity

complex spherical harmonics: inversion parity + total angular momentum conservation

In [5]:
@time coulomb_integral(Expand(), wf,wf,(1,0,0),(1,0,0),(1,0,0),(1,1,0);
                    R=π, SH_basis=:complex) |> println
@time coulomb_integral(Expand(), wf,wf,(1,0,0),(1,0,0),(1,0,0),(1,1,0);
                    R=π, SH_basis=:real) |> println
@time coulomb_integral(MonteCarlo(100000), wf,wf,(1,0,0),(1,0,0),(1,0,0),(1,1,0);
                    R=π, SH_basis=:complex) |> println
@time coulomb_integral(MonteCarlo(100000), wf,wf,(1,0,0),(1,0,0),(1,0,0),(1,1,0);
                    R=π, SH_basis=:real) |> println

(int = 0, err = 0)
  0.058371 seconds (48.57 k allocations: 3.560 MiB, 11.53% gc time, 98.72% compilation time)
(int = 0, err = 0)
  0.001111 seconds (76 allocations: 2.438 KiB)
(int = 0.0, err = 0.0, agg = CoulombIntegral.MonteCarloModule.Aggregator(5.890115434381606e-20 + 0.0im, 8.132394324532038e-37, 100000))
  0.004834 seconds (23.76 k allocations: 1.142 MiB)
(int = 0.0, err = 0.0, agg = CoulombIntegral.MonteCarloModule.Aggregator(4.9612636735138825e-20, 1.395104871862122e-36, 100000))
  0.011326 seconds (88.02 k allocations: 4.012 MiB)


### Method keyword arguments:

Expand: `kwargs...` are passed to each `hcubature` call

In [6]:
@time coulomb_integral(Expand(), wf,wf,(1,0,0),(1,1,1),(1,0,0),(1,1,1);
                    R=π, SH_basis=:real) |> println
@time coulomb_integral(Expand(; atol=1e-9), wf,wf,(1,0,0),(1,1,1),(1,0,0),(1,1,1);
                    R=π, SH_basis=:real) |> println

(int = 8.974001259406977, err = 1.3369370041628226e-7)
  0.015350 seconds (19.12 k allocations: 1.652 MiB, 42.66% compilation time)
(int = 8.974001241024519, err = 9.999781046292697e-10)
  0.174748 seconds (288.06 k allocations: 22.475 MiB, 48.12% compilation time)


note that in this case, the expansion has a single non-zero term with angular part equal to 1

MonteCarlo: `seed`

In [7]:
@time coulomb_integral(MonteCarlo(10000, seed=1), wf,wf,(1,0,0),(1,1,1),(1,0,0),(1,1,1);
                    R=π, SH_basis=:real) |> println
@time coulomb_integral(MonteCarlo(10000, seed=1), wf,wf,(1,0,0),(1,1,1),(1,0,0),(1,1,1);
                    R=π, SH_basis=:real) |> println

(int = 9.130391766170689, err = 0.2927218475685708, agg = CoulombIntegral.MonteCarloModule.Aggregator(0.0005412687548549418, 0.030110220373298604, 10000))
  0.200640 seconds (1.22 M allocations: 53.510 MiB, 6.09% gc time, 22.19% compilation time)
(int = 9.130391766170689, err = 0.2927218475685708, agg = CoulombIntegral.MonteCarloModule.Aggregator(0.0005412687548549418, 0.030110220373298604, 10000))
  0.139815 seconds (1.21 M allocations: 52.988 MiB, 2.93% gc time)


### MC: reusing aggregated results
keyword argument
`start::NamedTuple{(:M,:S,:N),Tuple{Number,Real,Integer}}=(M = 0, S = 0, N = 0)`


In [8]:
@time ci1 = coulomb_integral(MonteCarlo(100000), wf,wf,(1,0,0),(1,1,1),(1,0,0),(1,1,1);
                    R=π, SH_basis=:complex);
ci1 |> println
@time ci2 = coulomb_integral(MonteCarlo(100000), wf,wf,(1,0,0),(1,1,1),(1,0,0),(1,1,1);
                    R=π, SH_basis=:complex, start=ci1.agg);
ci2 |> println

  1.324615 seconds (11.62 M allocations: 529.063 MiB, 3.58% gc time)
(int = 8.945909860032884 + 0.0im, err = 0.06445497792123674, agg = CoulombIntegral.MonteCarloModule.Aggregator(0.0005303322809132158 + 0.0im, 0.1460009479083241, 100000))
  0.015087 seconds (2.65 k allocations: 190.336 KiB, 98.92% compilation time)
(int = 8.945909860032884 + 0.0im, err = 0.06445497792123674, agg = CoulombIntegral.MonteCarloModule.Aggregator(0.0005303322809132158 + 0.0im, 0.1460009479083241, 100000))
